# Bitcoin AML Detection on the Elliptic Dataset
### GraphSAGE vs. tabular baselines, with an honest temporal split

**Runs on:** Amazon SageMaker Studio (JupyterLab space) · data and outputs in Amazon S3

```
S3 (raw CSVs)  →  SageMaker (train + evaluate)  →  S3 (model + predictions)  →  UI
```

**Pipeline:** load 3 CSVs from S3 → build a transaction graph → temporal train/val/test split → GraphSAGE → XGBoost / Random Forest / hybrid baselines → comparison table → per-week drift chart → export model + risk-scored predictions back to S3.

Every code cell is preceded by a short note on **what** it does and **why** it's done that way.

> **Cost reminder:** a SageMaker space bills for every hour it is running, even idle. Stop it when you're done (see the last cell).

## Step 1 — Environment

`torch_geometric` (PyG) gives us `SAGEConv` and graph utilities. XGBoost gives the tree baseline. PyTorch, pandas, scikit-learn and `boto3` (the AWS Python library) already come with the SageMaker Distribution image.

**Why `%pip` and not `!pip`?** In SageMaker the notebook kernel can run in a different Python environment than the shell. `%pip` always installs into the kernel that's actually running your code; `!pip` sometimes installs somewhere else, and the import then fails.

**Why not install `torch_scatter`/`torch_sparse` too?** Modern PyG runs `SAGEConv` on plain PyTorch ops. Skipping those compiled extras avoids the most common install failure: a package built for a different CUDA/torch version.

In [ ]:
%pip install -q torch_geometric xgboost

## Step 2 — Imports, seed, device

A fixed seed makes runs reproducible: judges (and you) get the same numbers twice. GNN training on GPU is not *perfectly* deterministic, but the seed removes most of the variance.

In [ ]:
import os, json, random
import boto3
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import to_undirected

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (precision_recall_curve, auc, f1_score,
                             precision_score, recall_score, average_precision_score)
from xgboost import XGBClassifier

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Step 3 — Load the three CSVs from S3

The CSVs live in S3 at `s3://<BUCKET>/elliptic/data/`. We download them once to the space's local disk, then read them with pandas.

**Why download instead of reading `s3://…` directly?** pandas can read S3 paths, but only with the extra `s3fs` package. The features file is ~650 MB; downloading it once and skipping the download on reruns (`if not os.path.exists`) makes every rerun fast.

**Why no access keys in the code?** SageMaker gives the notebook an *execution role*, an AWS identity with permissions attached. `boto3` picks it up automatically, so there are no secrets to accidentally push to GitHub.

**Why the bucket name contains `sagemaker`:** the default SageMaker role (`AmazonSageMakerFullAccess`) is only allowed to read and write buckets whose names contain "sagemaker". Naming the bucket that way means no permission setup at all.

**Gotcha #1:** `elliptic_txs_features.csv` has **no header row**. Its 167 columns are: `txId`, `time_step`, 93 *local* features (facts about the transaction itself: fees, inputs, outputs…), and 72 *aggregated* features (one-hop neighbour summaries). We name them so we can select "local only" later.

**Gotcha #2:** the class file stores labels as strings (`"1"`, `"2"`, `"unknown"`). We cast to `str` before mapping so it works whether pandas read them as ints or strings.

In [ ]:
BUCKET   = 'sagemaker-brian-elliptic-aml'   # <-- your bucket name (keep 'sagemaker' in it)
PREFIX   = 'elliptic'                        # folder inside the bucket
DATA_DIR = 'data'                            # local folder on the SageMaker disk
CSV_FILES = ['elliptic_txs_features.csv', 'elliptic_txs_classes.csv', 'elliptic_txs_edgelist.csv']

s3 = boto3.client('s3')
os.makedirs(DATA_DIR, exist_ok=True)
for fn in CSV_FILES:
    local_path = f'{DATA_DIR}/{fn}'
    if os.path.exists(local_path):
        print('cached     ', fn)
    else:
        s3.download_file(BUCKET, f'{PREFIX}/data/{fn}', local_path)
        print('downloaded ', fn)

feats  = pd.read_csv(f'{DATA_DIR}/elliptic_txs_features.csv', header=None)
classes = pd.read_csv(f'{DATA_DIR}/elliptic_txs_classes.csv')
edges   = pd.read_csv(f'{DATA_DIR}/elliptic_txs_edgelist.csv')

LOCAL_COLS = [f'local_{i}' for i in range(93)]
AGG_COLS   = [f'agg_{i}'   for i in range(72)]
feats.columns = ['txId', 'time_step'] + LOCAL_COLS + AGG_COLS
FEATURE_COLS = LOCAL_COLS + AGG_COLS          # 165 model inputs (time_step is NOT a feature, see Step 4)

classes.columns = ['txId', 'class']
edges.columns   = ['txId1', 'txId2']

print('nodes:', len(feats), '| edges:', len(edges))
print(classes['class'].astype(str).value_counts())

## Step 4 — Labels, contiguous node indices, graph object

**Label mapping:** illicit → `1`, licit → `0`, unknown → `-1`. The `-1` nodes stay *in the graph* (their features and edges still help message passing) but are excluded from every loss and metric.

**Re-indexing:** PyG needs nodes numbered `0..N-1`. Raw txIds are arbitrary integers, so we build a `txId → row` dictionary and remap both edge columns through it.

**Undirected edges:** the edge list is directed (money flows A→B). We add the reverse edge so a node aggregates from *both* where its money came from and where it went. Laundering is about forwarding and splitting funds, so downstream neighbours carry signal too.

**Why `time_step` is not a feature:** the test weeks (35–49) never appear in training, so a model can't learn anything useful from the raw week number. Worse, tree models could latch onto it. We keep it as metadata for splitting and charts only.

In [ ]:
df = feats.merge(classes, on='txId', how='left')
df['y'] = df['class'].astype(str).map({'1': 1, '2': 0, 'unknown': -1}).fillna(-1).astype(int)

id2idx = {tx: i for i, tx in enumerate(df['txId'].values)}
src = edges['txId1'].map(id2idx)
dst = edges['txId2'].map(id2idx)
ok  = src.notna() & dst.notna()
print(f'dropped {(~ok).sum()} edges with unknown endpoints')

edge_index = torch.tensor(np.vstack([src[ok].values, dst[ok].values]).astype(np.int64))
edge_index = to_undirected(edge_index)

data = Data(
    x          = torch.tensor(df[FEATURE_COLS].values, dtype=torch.float),
    edge_index = edge_index,
    y          = torch.tensor(df['y'].values, dtype=torch.long),
    time_step  = torch.tensor(df['time_step'].values, dtype=torch.long),
)
data.txid = torch.tensor(df['txId'].values, dtype=torch.long)
print(data)

## Step 5 — Temporal split, class weight, feature scaling

| Split | Weeks | Purpose |
|---|---|---|
| train | 1–29 | fit weights |
| val   | 30–34 | early stopping + threshold tuning |
| test  | 35–49 | reported **once**, at the end |

**Why temporal, not random?** In production the model is trained on the past and scores the future. A random split lets it train on week-40 transactions and test on their week-40 neighbours: leakage that inflates every metric.

**Why a validation window?** We must make decisions (which epoch to keep, which probability cutoff to use). Making them on the test set would be tuning to the answers. The val window sits *after* train in time, so it also mimics the real train→test gap.

**`pos_weight`** = licit/illicit ratio in train (≈9). `BCEWithLogitsLoss` then charges ~9× more for a missed illicit transaction, so the model can't cheat by predicting "licit" for everyone.

**Scaling:** features span wildly different ranges. GraphSAGE *averages* neighbour vectors, so big-scale columns drown small ones. `StandardScaler` is fit on **train rows only**: computing a mean over test rows would leak test statistics into training.

In [ ]:
t, y = data.time_step, data.y
labeled = y != -1
train_mask = (t <= 29) & labeled
val_mask   = (t >= 30) & (t <= 34) & labeled
test_mask  = (t >= 35) & labeled
print(f'train {train_mask.sum().item():,} | val {val_mask.sum().item():,} | test {test_mask.sum().item():,}')

n_illicit = (y[train_mask] == 1).sum().item()
n_licit   = (y[train_mask] == 0).sum().item()
pos_weight = torch.tensor([n_licit / n_illicit], dtype=torch.float)
print(f'train illicit rate {n_illicit/(n_illicit+n_licit):.2%}  ->  pos_weight = {pos_weight.item():.2f}')

scaler = StandardScaler().fit(data.x[train_mask].numpy())
X_scaled = scaler.transform(data.x.numpy()).astype(np.float32)
data.x = torch.tensor(X_scaled)

data = data.to(device)
train_mask, val_mask, test_mask = train_mask.to(device), val_mask.to(device), test_mask.to(device)
pos_weight = pos_weight.to(device)

## Step 6 — Shared evaluation helpers

* `best_threshold` picks the probability cutoff that maximises illicit-F1 **on validation**. Under `pos_weight` the model's probabilities are pushed upward, so 0.5 is no longer meaningful.
* `metrics` returns exactly the four numbers in the comparison table. No accuracy: a model that flags nothing scores ~98% and catches no one.
* PR-AUC is computed from the precision-recall curve (and `average_precision_score` is printed alongside as a sanity check: the two should be close).

Every model, GNN or tree, goes through these same two functions, so the comparison is fair.

In [ ]:
def best_threshold(y_true, probs):
    p, r, thr = precision_recall_curve(y_true, probs)
    f1 = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-9, None)
    return float(thr[np.argmax(f1)])

def metrics(y_true, probs, thr):
    pred = (probs >= thr).astype(int)
    p, r, _ = precision_recall_curve(y_true, probs)
    return {
        'Illicit F1':        f1_score(y_true, pred, pos_label=1),
        'Illicit Recall':    recall_score(y_true, pred, pos_label=1),
        'Illicit Precision': precision_score(y_true, pred, pos_label=1, zero_division=0),
        'PR-AUC':            auc(r, p),
        'AvgPrecision':      average_precision_score(y_true, probs),
        'threshold':         thr,
    }

def per_timestep_f1(y_true, probs, thr, steps):
    out = {}
    for s in sorted(np.unique(steps)):
        m = steps == s
        out[int(s)] = f1_score(y_true[m], (probs[m] >= thr).astype(int), pos_label=1, zero_division=0)
    return out

results = {}     # model name -> metrics dict
test_probs = {}  # model name -> probabilities on test nodes (for the drift chart)

y_val  = data.y[val_mask].cpu().numpy()
y_test = data.y[test_mask].cpu().numpy()
t_test = data.time_step[test_mask].cpu().numpy()

## Step 7 — GraphSAGE model

Exactly the spec: `SAGEConv(165→128) → ReLU → Dropout → SAGEConv(128→64) → ReLU → Dropout → Linear(64→1)`.

**How SAGEConv works (one sentence for judges):** each layer updates a node by combining its own vector with the *mean* of its neighbours' vectors, so after two layers every transaction "knows" about its 2-hop neighbourhood.

`embed()` returns the 64-d vector *before* the classifier head. We reuse it later to feed graph knowledge into XGBoost (the hybrid model).

In [ ]:
class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden=128, out=64, dropout=0.3):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden)
        self.conv2 = SAGEConv(hidden, out)
        self.head  = nn.Linear(out, 1)
        self.dropout = dropout

    def embed(self, x, edge_index):
        x = F.dropout(F.relu(self.conv1(x, edge_index)), self.dropout, self.training)
        x = F.dropout(F.relu(self.conv2(x, edge_index)), self.dropout, self.training)
        return x

    def forward(self, x, edge_index):
        return self.head(self.embed(x, edge_index)).squeeze(-1)

model = GraphSAGE(data.num_features).to(device)
print(model)

## Step 8 — Train with early stopping

Full-batch training: the whole graph (203k nodes) fits on a T4, so every epoch is one forward pass. No mini-batching needed.

**Model selection by validation PR-AUC**, not by loss and not by test score. PR-AUC is threshold-free, so we don't have to commit to a cutoff mid-training. We keep the weights from the best validation epoch and stop after 15 epochs without improvement: GNNs overfit, and the last epoch is rarely the best one.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

EPOCHS, PATIENCE = 100, 15
best_ap, best_state, wait = -1, None, 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x, data.edge_index)
    loss = criterion(logits[train_mask], data.y[train_mask].float())
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_p = torch.sigmoid(model(data.x, data.edge_index)[val_mask]).cpu().numpy()
    ap = average_precision_score(y_val, val_p)

    if ap > best_ap:
        best_ap, wait = ap, 0
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
    else:
        wait += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'epoch {epoch:3d} | loss {loss.item():.4f} | val AP {ap:.4f}')
    if wait >= PATIENCE:
        print(f'early stop at epoch {epoch}'); break

model.load_state_dict(best_state)
print(f'best epoch {best_epoch} | val AP {best_ap:.4f}')

## Step 9 — Evaluate GraphSAGE on the test weeks

Threshold is chosen on **val**, then frozen and applied to **test**. This is the one and only time the test set is scored for this model.

In [ ]:
model.eval()
with torch.no_grad():
    all_p = torch.sigmoid(model(data.x, data.edge_index)).cpu().numpy()

thr = best_threshold(y_val, all_p[val_mask.cpu().numpy()])
gnn_test_p = all_p[test_mask.cpu().numpy()]
results['GraphSAGE'] = metrics(y_test, gnn_test_p, thr)
test_probs['GraphSAGE'] = gnn_test_p
pd.DataFrame(results).T

## Step 10 — Tabular baselines on the *same* split

Same scaled features, same train/val/test weeks, same threshold-tuning routine. Only the model changes.

* **XGBoost (all 165 features)** — `scale_pos_weight` is the tree equivalent of `pos_weight`; early stopping on val `aucpr`.
* **XGBoost (93 local features)** — the 72 aggregated columns are hand-built neighbour summaries, i.e. graph information baked into a table. Removing them shows how much the *neighbourhood* matters versus the transaction alone.
* **Random Forest** — the strongest single baseline in the original Elliptic paper (Weber et al., 2019), where it beat the GCN. Expect it to be hard to beat.
* **Hybrid: GNN embeddings + raw features → XGBoost** — trees are good at sharp per-feature rules, the GNN is good at structure. Feeding the 64-d SAGE embedding into XGBoost combines both. No leakage: embeddings are computed from features and edges, never from test labels.

In [ ]:
X_all  = data.x.cpu().numpy()
tr, va, te = train_mask.cpu().numpy(), val_mask.cpu().numpy(), test_mask.cpu().numpy()
y_all  = data.y.cpu().numpy()
spw    = float(pos_weight.item())

def run_xgb(name, X):
    clf = XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric='aucpr', early_stopping_rounds=50,
        tree_method='hist', device='cuda' if device.type == 'cuda' else 'cpu',
        random_state=SEED,
    )
    clf.fit(X[tr], y_all[tr], eval_set=[(X[va], y_all[va])], verbose=False)
    thr = best_threshold(y_val, clf.predict_proba(X[va])[:, 1])
    p = clf.predict_proba(X[te])[:, 1]
    results[name] = metrics(y_test, p, thr); test_probs[name] = p
    return clf

xgb_all   = run_xgb('XGBoost (all features)', X_all)
xgb_local = run_xgb('XGBoost (local only)', X_all[:, :len(LOCAL_COLS)])

rf = RandomForestClassifier(n_estimators=300, max_features=50, class_weight='balanced_subsample',
                            n_jobs=-1, random_state=SEED)
rf.fit(X_all[tr], y_all[tr])
thr_rf = best_threshold(y_val, rf.predict_proba(X_all[va])[:, 1])
p_rf = rf.predict_proba(X_all[te])[:, 1]
results['Random Forest'] = metrics(y_test, p_rf, thr_rf); test_probs['Random Forest'] = p_rf

model.eval()
with torch.no_grad():
    emb = model.embed(data.x, data.edge_index).cpu().numpy()
X_hybrid = np.hstack([X_all, emb])
xgb_hybrid = run_xgb('Hybrid: SAGE emb + XGBoost', X_hybrid)

pd.DataFrame(results).T

## Step 11 — Comparison table

The deliverable from the spec. Sorted by illicit F1. Saved to `results.csv`.

In [ ]:
table = (pd.DataFrame(results).T[['Illicit F1', 'Illicit Recall', 'Illicit Precision', 'PR-AUC']]
           .sort_values('Illicit F1', ascending=False).round(4))
table.index.name = 'Model'
table.to_csv('results.csv')
print(table.to_markdown())

## Step 12 — Per-week F1: the concept-drift story

One aggregate number hides the most interesting thing about this dataset. Around **time step 43 a major dark market was shut down**; criminal behaviour changed and every model's recall collapses on later weeks. Showing this proves the model isn't broken: the *world* changed, which is exactly why real AML systems are retrained continuously.

In [ ]:
plt.figure(figsize=(10, 4.5))
for name, p in test_probs.items():
    f1s = per_timestep_f1(y_test, p, results[name]['threshold'], t_test)
    plt.plot(list(f1s.keys()), list(f1s.values()), marker='o', label=name)
plt.axvline(43, color='k', ls='--', lw=1); plt.text(43.2, 0.95, 'dark market\nshutdown', fontsize=9)
plt.xlabel('time step (week)'); plt.ylabel('illicit F1'); plt.ylim(0, 1)
plt.title('Illicit F1 per test week'); plt.legend(fontsize=8); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig('f1_per_timestep.png', dpi=150); plt.show()

## Step 13 — Export artifacts (local disk, then S3)

Files are written to the SageMaker disk first, then uploaded to `s3://<BUCKET>/elliptic/outputs/`. S3 is the durable copy: the space's disk goes away if the space is deleted, and your UI teammates can pull the JSON straight from the bucket.

**`elliptic_graphsage.pt`** contains the weights *plus* everything needed to reproduce a prediction from raw features: scaler mean/std, the tuned threshold, and input size. Weights alone would be useless to a UI.

**`sample_predictions.json`** is stratified, not random. A random sample of test nodes is ~98% licit, which makes for a dull demo. We include true positives, true negatives, false positives and false negatives so the UI shows real behaviour, warts and all. `risk_score` is `prob × 100`; `risk_tier` mirrors how analysts triage an alert queue.

In [ ]:
torch.save({
    'state_dict':   model.state_dict(),
    'in_channels':  data.num_features,
    'scaler_mean':  scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'threshold':    results['GraphSAGE']['threshold'],
    'feature_cols': FEATURE_COLS,
}, 'elliptic_graphsage.pt')

thr_g = results['GraphSAGE']['threshold']
idx_test = np.where(te)[0]
pred = (gnn_test_p >= thr_g).astype(int)
groups = {'TP': (y_test==1)&(pred==1), 'FN': (y_test==1)&(pred==0),
          'FP': (y_test==0)&(pred==1), 'TN': (y_test==0)&(pred==0)}
rng = np.random.default_rng(SEED)
chosen = np.concatenate([rng.choice(np.where(m)[0], size=min(25, m.sum()), replace=False)
                         for m in groups.values() if m.sum() > 0])

def tier(p):
    return 'high' if p >= thr_g else 'medium' if p >= thr_g / 2 else 'low'

txid_np = data.txid.cpu().numpy()
sample = [{
    'txId':        int(txid_np[idx_test[i]]),
    'time_step':   int(t_test[i]),
    'true_label':  int(y_test[i]),
    'prob_illicit': round(float(gnn_test_p[i]), 4),
    'predicted':   int(pred[i]),
    'risk_score':  round(float(gnn_test_p[i]) * 100, 1),
    'risk_tier':   tier(float(gnn_test_p[i])),
} for i in chosen]

with open('sample_predictions.json', 'w') as f:
    json.dump({'threshold': thr_g, 'n': len(sample), 'predictions': sample}, f, indent=2)
print(f'wrote {len(sample)} predictions;', {k: int(v.sum()) for k, v in groups.items()})

OUT_FILES = ['elliptic_graphsage.pt', 'sample_predictions.json', 'results.csv', 'f1_per_timestep.png']
for fn in OUT_FILES:
    s3.upload_file(fn, BUCKET, f'{PREFIX}/outputs/{fn}')
    print(f'uploaded s3://{BUCKET}/{PREFIX}/outputs/{fn}')

## Step 14 — Share the predictions with the UI (optional)

A **presigned URL** is a temporary link that lets anyone with the link download one S3 file, without an AWS account and without making the bucket public.

It's signed with the notebook role's temporary credentials, so it stops working when those credentials expire (usually within a few hours), even if `ExpiresIn` is longer. Rerun this cell to get a fresh link.

In [ ]:
url = s3.generate_presigned_url(
    'get_object',
    Params={'Bucket': BUCKET, 'Key': f'{PREFIX}/outputs/sample_predictions.json'},
    ExpiresIn=6 * 3600,
)
print(url)

## Done: stop the space

In Studio: **JupyterLab → your space → Stop**. A stopped space keeps its files but stops billing for compute. Your results are also safe in S3.